# Coltrain v3.1 — Jazz Music Transformer Training

**Model**: 26M params (d_model=512, 8 layers, 8 heads, d_ff=2048, seq_len=2048)

**Dataset**: `jazz_v3_2k.npy` — 308,848 sequences × 2048 tokens (1.18 GB)

**Format**: `[BOS][CHORDS]chords...[LEAD][COMPING][BASS][DRUMS][EOS]` — chord-conditioned multi-track

**Training**: Single shared model, standard CE loss, 5 epochs, cosine LR with warmup

---

**Before running:**
1. Runtime → Change runtime type → **A100 GPU** (or L4 if A100 unavailable)
2. Upload these files to Google Drive root (`/content/drive/MyDrive/`):
   - `jazz_v3_2k.npy` (1.18 GB)
   - `jazz_v3_2k_groups.json` (8.5 MB)
3. Run all cells — training auto-resumes if Colab disconnects

In [ ]:
# Cell 1: Mount Drive + Check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({mem:.0f} GB)')
    # Detect GPU tier for batch size
    if 'A100' in gpu:
        GPU_TIER = 'A100'  # 40-80GB
    elif 'L4' in gpu:
        GPU_TIER = 'L4'    # 24GB
    elif 'V100' in gpu:
        GPU_TIER = 'V100'  # 16GB
    elif 'T4' in gpu:
        GPU_TIER = 'T4'    # 16GB
    else:
        GPU_TIER = 'other'
    print(f'GPU tier: {GPU_TIER}')
else:
    raise RuntimeError('No GPU! Change runtime type to GPU.')

In [ ]:
# Cell 2: Upload code from this repo
# Option A: Clone from GitHub (if you pushed the repo)
# !git clone https://github.com/YOUR_USER/coltrain-april-9.git /content/coltrain

# Option B: Upload code zip from Drive
import os, zipfile, shutil

DRIVE = '/content/drive/MyDrive'
WORK = '/content/coltrain'

# Try GitHub clone first, fall back to Drive zip
if not os.path.exists(WORK):
    zip_path = f'{DRIVE}/coltrain_code_v3.zip'
    if os.path.exists(zip_path):
        print('Extracting code from Drive...')
        os.makedirs(WORK, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(WORK)
        print('Done!')
    else:
        raise FileNotFoundError(
            f'No code found! Either:\n'
            f'  1. Clone from GitHub (uncomment line above)\n'
            f'  2. Upload coltrain_code_v3.zip to Google Drive root'
        )
else:
    print(f'{WORK} already exists')

# Verify key files
for f in ['train_v3.py', 'generate_v3.py', 'transformer/model/transformer.py',
          'transformer/data/tokenizer_v3.py', 'transformer/data/dataset.py']:
    assert os.path.exists(f'{WORK}/{f}'), f'Missing: {f}'
print('All code files verified!')

In [ ]:
# Cell 3: Check dataset
import numpy as np

# Dataset can be on Drive or local
npy_path = f'{DRIVE}/jazz_v3_2k.npy'
groups_path = f'{DRIVE}/jazz_v3_2k_groups.json'

# Copy to local disk for faster I/O (Drive is slow for random reads)
local_npy = '/content/jazz_v3_2k.npy'
local_groups = '/content/jazz_v3_2k_groups.json'

if not os.path.exists(local_npy):
    print('Copying dataset to local disk (faster I/O)...')
    shutil.copy(npy_path, local_npy)
    shutil.copy(groups_path, local_groups)
    print('Done!')

data = np.load(local_npy, mmap_mode='r')
print(f'Dataset: {data.shape[0]:,} sequences × {data.shape[1]} tokens')
print(f'Size: {os.path.getsize(local_npy) / 1e9:.2f} GB')

In [ ]:
# Cell 4: Configuration
import sys
sys.path.insert(0, f'{WORK}/transformer')

# Auto-tune batch size to GPU
BATCH_SIZES = {
    'A100': 64,   # 40GB+ VRAM
    'L4':   36,   # 24GB VRAM
    'V100': 32,   # 16GB VRAM
    'T4':   24,   # 16GB VRAM (less efficient)
    'other': 16,
}

# Use bfloat16 on Ampere+ (A100, L4), float16 on older (V100, T4)
AMP_DTYPE = {
    'A100': torch.bfloat16,
    'L4':   torch.bfloat16,
    'V100': torch.float16,
    'T4':   torch.float16,
    'other': torch.float16,
}

batch_size = BATCH_SIZES.get(GPU_TIER, 16)
amp_dtype = AMP_DTYPE.get(GPU_TIER, torch.float16)

CONFIG = {
    # Model (26M params)
    'd_model': 512,
    'num_heads': 8,
    'num_layers': 8,
    'd_ff': 2048,
    'max_seq_len': 2048,
    'dropout': 0.2,
    # Data
    'pretokenized_path': local_npy,
    'groups_path': local_groups,
    # Training
    'batch_size': batch_size,
    'num_epochs': 5,
    'learning_rate': 3e-4,
    'weight_decay': 0.05,
    'warmup_steps': 1000,
    'max_grad_norm': 1.0,
    'val_ratio': 0.05,
    'label_smoothing': 0.1,
    # AMP
    'amp_dtype': amp_dtype,
}

print(f'GPU: {GPU_TIER} → batch_size={batch_size}, amp={amp_dtype}')
print(f'\nFull config:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# Cell 5: Create model + data loaders
import math
import time
import json
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from model.transformer import MusicTransformer
from data.tokenizer_v3 import TrackSeqTokenizer
from data.dataset import create_pretrain_dataloader

tokenizer = TrackSeqTokenizer()
print(f'Vocab: {tokenizer.vocab_size}')

train_loader, val_loader = create_pretrain_dataloader(
    bin_path=CONFIG['pretokenized_path'],
    seq_len=CONFIG['max_seq_len'],
    batch_size=CONFIG['batch_size'],
    num_workers=4,
    val_ratio=CONFIG['val_ratio'],
    groups_path=CONFIG['groups_path'],
)

model = MusicTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=CONFIG['d_model'],
    num_heads=CONFIG['num_heads'],
    num_layers=CONFIG['num_layers'],
    d_ff=CONFIG['d_ff'],
    max_seq_len=CONFIG['max_seq_len'],
    dropout=CONFIG['dropout'],
).cuda()

criterion = nn.CrossEntropyLoss(
    ignore_index=tokenizer.pad_id,
    label_smoothing=CONFIG['label_smoothing'],
)
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    betas=(0.9, 0.98),
    weight_decay=CONFIG['weight_decay'],
)

print(f'\nTraining batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')
print(f'Steps per epoch: {len(train_loader)}')
print(f'Total steps: {len(train_loader) * CONFIG["num_epochs"]}')

In [ ]:
# Cell 6: Resume from checkpoint (if Colab reconnected)
CKPT_DIR = Path(f'{WORK}/checkpoints_v3')
CKPT_DIR.mkdir(exist_ok=True)
PREFIX = 'v3_shared'

start_epoch = 1
best_val_loss = float('inf')
current_step = 0

# Check for resume checkpoint (local first, then Drive)
resume_path = CKPT_DIR / f'{PREFIX}_latest.pt'
drive_resume = f'{DRIVE}/{PREFIX}_latest.pt'

if resume_path.exists():
    ckpt_path = resume_path
elif os.path.exists(drive_resume):
    print(f'Restoring checkpoint from Drive...')
    shutil.copy(drive_resume, resume_path)
    ckpt_path = resume_path
else:
    ckpt_path = None

if ckpt_path:
    print(f'Resuming from {ckpt_path}...')
    ckpt = torch.load(ckpt_path, map_location='cuda', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['val_loss']
    current_step = ckpt.get('current_step', 0)
    print(f'Resumed: epoch {start_epoch}, best val_loss={best_val_loss:.4f}, step={current_step}')
else:
    print('Starting fresh training')

In [ ]:
# Cell 7: Training loop
steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * CONFIG['num_epochs']
warmup = CONFIG['warmup_steps']
base_lr = CONFIG['learning_rate']
use_amp = True
amp_dtype = CONFIG['amp_dtype']
use_scaler = (amp_dtype == torch.float16)  # GradScaler only for fp16, not bf16
scaler = torch.amp.GradScaler('cuda') if use_scaler else None

def update_lr(step):
    if step < warmup:
        lr = base_lr * (step / max(warmup, 1))
    else:
        progress = (step - warmup) / max(total_steps - warmup, 1)
        lr = base_lr * 0.5 * (1.0 + math.cos(min(progress, 1.0) * math.pi))
        lr = max(lr, base_lr * 0.01)
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    return lr

def save_and_backup(ckpt, name):
    """Save checkpoint locally and to Drive."""
    torch.save(ckpt, CKPT_DIR / name)
    shutil.copy(CKPT_DIR / name, f'{DRIVE}/{name}')

print(f'Training: epochs {start_epoch}-{CONFIG["num_epochs"]}')
print(f'AMP: {amp_dtype}, GradScaler: {use_scaler}')
print(f'Steps/epoch: {steps_per_epoch}, Total: {total_steps}')
print('=' * 70)

for epoch in range(start_epoch, CONFIG['num_epochs'] + 1):
    epoch_start = time.time()
    model.train()
    total_loss = 0
    total_batches = 0

    for batch_idx, (inp, tgt) in enumerate(train_loader):
        inp = inp.cuda(non_blocking=True)
        tgt = tgt.cuda(non_blocking=True)

        with torch.amp.autocast('cuda', dtype=amp_dtype):
            logits = model(inp)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))

        if use_scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
            optimizer.step()

        optimizer.zero_grad(set_to_none=True)
        lr = update_lr(current_step)
        current_step += 1

        total_loss += loss.item()
        total_batches += 1

        if (batch_idx + 1) % 100 == 0:
            avg = total_loss / total_batches
            ppl = min(math.exp(avg), 99999)
            elapsed = time.time() - epoch_start
            eta = elapsed / (batch_idx + 1) * (steps_per_epoch - batch_idx - 1)
            mem = torch.cuda.max_memory_allocated() / 1e9
            print(f'  [{batch_idx+1}/{steps_per_epoch}] '
                  f'loss={avg:.4f} ppl={ppl:.1f} lr={lr:.2e} '
                  f'VRAM={mem:.1f}GB ETA={eta/60:.0f}min')

    # Validate
    model.eval()
    val_loss = 0
    val_batches = 0
    with torch.no_grad():
        for inp, tgt in val_loader:
            inp = inp.cuda(non_blocking=True)
            tgt = tgt.cuda(non_blocking=True)
            with torch.amp.autocast('cuda', dtype=amp_dtype):
                logits = model(inp)
                loss = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
            val_loss += loss.item()
            val_batches += 1

    train_avg = total_loss / total_batches
    val_avg = val_loss / max(val_batches, 1)
    elapsed = (time.time() - epoch_start) / 60

    print(f'\nEpoch {epoch}/{CONFIG["num_epochs"]} ({elapsed:.1f} min):')
    print(f'  Train: {train_avg:.4f} (ppl {min(math.exp(train_avg), 99999):.1f})')
    print(f'  Val:   {val_avg:.4f} (ppl {min(math.exp(val_avg), 99999):.1f})')

    # Save checkpoint
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_avg,
        'current_step': current_step,
        'config': {k: v for k, v in CONFIG.items() if k != 'amp_dtype'},
        'tokenizer_type': 'v3_trackseq',
        'role': 'shared',
    }
    save_and_backup(ckpt, f'{PREFIX}_epoch{epoch}.pt')
    save_and_backup(ckpt, f'{PREFIX}_latest.pt')
    print(f'  Saved checkpoint to Drive')

    if val_avg < best_val_loss:
        best_val_loss = val_avg
        save_and_backup(ckpt, f'{PREFIX}_best.pt')
        print(f'  ★ New best! val_loss={best_val_loss:.4f}')

print(f'\nTraining complete! Best val loss: {best_val_loss:.4f}')
print(f'Checkpoint saved to: {DRIVE}/{PREFIX}_best.pt')

In [ ]:
# Cell 8: Quick generation test (after training)
sys.path.insert(0, WORK)
from generate_v3 import (
    load_model, generate_cascaded, analyze_output,
    FORM_TEMPLATES, TrackSeqTokenizer
)

# Load best model
tok = TrackSeqTokenizer()
best_path = str(CKPT_DIR / f'{PREFIX}_best.pt')
shared = load_model(best_path, tok.vocab_size, 'cuda')
models = {r: shared for r in ['lead', 'comping', 'bass', 'drums']}

# Generate Giant Steps in B with full Coltrane features
template = FORM_TEMPLATES['giantsteps']
grid = template.build_grid(key_pc=11, num_choruses=2)  # B, 2 choruses

theory_params = {'chord_tone_boost': 2.0, 'scale_tone_boost': 1.0, 'avoid_penalty': 1.5}
grid_params = {'bass_root_boost': 3.0, 'approach_boost': 1.5, 'phrase_breathing': 1.5}
coltrane_params = {
    'tension_curve': True,
    'motivic_memory': True,
    'key_center_boost': 0.5,
    'arpeggio_boost': 2.0,
    'motif_boost': 1.5,
}

tokens = generate_cascaded(
    models=models, tokenizer=tok,
    temperature=0.9, top_k=50, top_p=0.95,
    theory_params=theory_params,
    beat_grid=grid, grid_params=grid_params,
    coltrane_params=coltrane_params,
    device='cuda',
)

analyze_output(tokens, tok)

# Save to Drive
output_path = f'{DRIVE}/generated_giantsteps_coltrane.mid'
tok.detokenize_to_midi(tokens, output_path, tempo=280)  # Giant Steps is fast!
print(f'\nSaved: {output_path}')

In [ ]:
# Cell 9: Jazz evaluation (optional)
sys.path.insert(0, WORK)
!python {WORK}/jazz_eval.py {output_path}